In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
from surprise import Dataset, Reader, accuracy
from surprise.model_selection import KFold
from models import UserBased_tuned
from loaders import load_ratings
from constants import Constant as C

# 1. Chargement des données
df = load_ratings()
reader = Reader(rating_scale=C.RATINGS_SCALE)
data = Dataset.load_from_df(df[[C.USER_ID_COL, C.ITEM_ID_COL, C.RATING_COL]], reader)

# Dictionnaire pour stocker les prédictions séparément
dict_predictions = {}

for sim_type in ['msd', 'jacard', 'cosine_jaccard']:
    print(f"\n--- Évaluation pour la similarité : {sim_type.upper()} ---")
    best_algo = UserBased_tuned(k=40, min_k=3, sim_options={'name': sim_type, 'min_support': 5})

    kf = KFold(n_splits=5, random_state=42)
    all_predictions = []

    print("Entraînement et génération des prédictions en cours...")
    for trainset, testset in kf.split(data):
        best_algo.fit(trainset)
        predictions = best_algo.test(testset)
        all_predictions.extend(predictions)

    # Sauvegarde dans notre dictionnaire général
    dict_predictions[sim_type] = all_predictions
    print(f"Prédictions stockées pour {sim_type}.")

# Extraction du trainset complet (commun aux deux)
full_trainset = data.build_full_trainset()


--- Évaluation pour la similarité : MSD ---
Entraînement et génération des prédictions en cours...
Prédictions stockées pour msd.

--- Évaluation pour la similarité : JACARD ---
Entraînement et génération des prédictions en cours...
Prédictions stockées pour jacard.

--- Évaluation pour la similarité : COSINE_JACCARD ---
Entraînement et génération des prédictions en cours...
Prédictions stockées pour cosine_jaccard.


In [2]:
def evaluate_accuracy_metrics(predictions):
    """Calcule et affiche le RMSE et la MAE."""
    print("--- Métriques d'Erreur Numérique ---")
    rmse_score = accuracy.rmse(predictions, verbose=False)
    mae_score = accuracy.mae(predictions, verbose=False)
    
    print(f"RMSE : {rmse_score:.4f}")
    print(f"MAE  : {mae_score:.4f}")
    return {"RMSE": round(rmse_score, 4), "MAE": round(mae_score, 4)}

# Test de la cellule
# accuracy_results = evaluate_accuracy_metrics(all_predictions)

In [4]:
def evaluate_novelty(predictions, trainset, k=10):
    """Calcule la nouveauté moyenne (self-information) des recommandations du Top-K."""
    print(f"--- Métriques de Comportement (Novelty@{k}) ---")
    
    n_users = trainset.n_users
    
    # 1. Calculer la popularité de chaque film dans le trainset (basé sur les INNER IDs)
    item_popularities = defaultdict(int)
    for u, i, _ in trainset.all_ratings():
        item_popularities[i] += 1
        
    # Transformer en auto-information : -log2(p(i))
    item_novelty = dict()
    for i in range(trainset.n_items):
        freq = item_popularities[i]
        prob = freq / n_users if n_users > 0 else 0
        item_novelty[i] = -np.log2(prob) if prob > 0 else 0

    # 2. Récupérer le Top-K estimé pour chaque utilisateur
    user_est_true = defaultdict(list)
    for uid, iid, _, est, _ in predictions:
        # Crucial : predictions contient des RAW IDs. On doit convertir l'iid en INNER ID.
        try:
            inner_iid = trainset.to_inner_iid(iid)
            user_est_true[uid].append((est, inner_iid))
        except ValueError:
            # Si le film n'est pas dans le trainset (cas rare de cold start d'item), on ignore
            continue
        
    total_novelty = 0
    user_count = 0
    
    for uid, user_ratings in user_est_true.items():
        # Trier par note estimée décroissante
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k_items = user_ratings[:k]
        
        if not top_k_items:
            continue
            
        # Moyenne de la nouveauté du Top-K de cet utilisateur
        user_novelty = np.mean([item_novelty[inner_id] for _, inner_id in top_k_items])
        total_novelty += user_novelty
        user_count += 1
        
    mean_novelty = total_novelty / user_count if user_count > 0 else 0
    
    print(f"Nouveauté Moyenne (Novelty@{k}) : {mean_novelty:.4f} bits")
    return {f"Novelty@{k}": round(mean_novelty, 4)}

In [5]:
def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """Calcule la Précision@K et le Rappel@K sur l'ensemble des utilisateurs."""
    print(f"--- Métriques de Classement (Top-{k} à un seuil de {threshold}) ---")
    
    # Regrouper les prédictions par utilisateur
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()

    for uid, user_ratings in user_est_true.items():
        # Trier les films par note estimée décroissante
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Nombre de films réellement pertinents pour cet utilisateur
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        # Nombre de films recommandés dans le Top-K qui sont jugés pertinents
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])

        # Nombre de films à la fois recommandés dans le Top-K ET réellement pertinents
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[:k])

        # Précision@K : Proportion de recommandations pertinentes
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

        # Rappel@K : Proportion de films aimés qui ont été capturés
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    # Moyenne générale sur tous les utilisateurs
    mean_precision = np.mean(list(precisions.values()))
    mean_recall = np.mean(list(recalls.values()))

    print(f"Precision@{k} : {mean_precision * 100:.2f}%")
    print(f"Recall@{k}    : {mean_recall * 100:.2f}%")
    
    return {f"Precision@{k}": round(mean_precision, 4), f"Recall@{k}": round(mean_recall, 4)}

# Test de la cellule
# classification_results = precision_recall_at_k(all_predictions, k=10, threshold=3.5)

In [6]:
def evaluate_catalog_coverage(predictions, trainset, k=10):
    """Calcule le pourcentage de films uniques recommandés au moins une fois."""
    print(f"--- Métriques de Qualité du Catalogue (Top-{k}) ---")
    
    user_est_true = defaultdict(list)
    for uid, iid, _, est, _ in predictions:
        user_est_true[uid].append((est, iid))
        
    recommended_items = set()
    
    for uid, user_ratings in user_est_true.items():
        # Trier par note estimée décroissante
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        # Prendre le top K et ajouter les IDs de films au Set
        for _, iid in user_ratings[:k]:
            recommended_items.add(iid)
            
    # Calcul du ratio : (Nb de films recommandés au moins une fois) / (Total de films dans le trainset)
    total_unique_items = trainset.n_items
    coverage = len(recommended_items) / total_unique_items
    
    print(f"Couverture du Catalogue : {coverage * 100:.2f}% ({len(recommended_items)} films sur {total_unique_items})")
    return {"Catalog Coverage": round(coverage, 4)}

# Test de la cellule
# coverage_results = evaluate_catalog_coverage(all_predictions, full_trainset, k=10)

In [7]:
print("==================================================")
print("   COMPARAISON GLOBALE : MSD VS JACCARD CUSTOM    ")
print("==================================================\n")

# Dictionnaire global pour stocker les rapports finaux
final_comparison_data = {}

# Boucle d'évaluation
for sim_type in ['msd', 'jacard', 'cosine_jaccard']:
    print(f"👉 Calcul de toutes les métriques pour : {sim_type.upper()}...")
    preds = dict_predictions[sim_type]
    
    # Appel de tes fonctions (sans affichage verbeux si tu veux épargner de l'espace)
    error_metrics = evaluate_accuracy_metrics(preds)
    ranking_metrics = precision_recall_at_k(preds, k=10, threshold=3.5)
    coverage_metrics = evaluate_catalog_coverage(preds, full_trainset, k=10)
    novelty_metrics = evaluate_novelty(preds, full_trainset, k=10)
    
    # Fusion des résultats pour ce modèle
    final_comparison_data[sim_type] = {**error_metrics, **ranking_metrics, **coverage_metrics, **novelty_metrics}
    print(f"✓ Calcul terminé pour {sim_type.upper()}.\n")

# 2. Construction du DataFrame de comparaison finale
# On transforme le dictionnaire imbriqué en tableau structuré
summary_df = pd.DataFrame(final_comparison_data)

# Renommer proprement les colonnes pour le rapport du prof
summary_df.columns = ['Valeur (User-Based MSD)', 'Valeur (User-Based Jaccard)', 'Valeur (User-Based Cosine Jaccard)']
summary_df.index.name = 'Métrique'
summary_df = summary_df.reset_index()

print("==================================================")
print("         TABLEAU COMPARATIF FINAL                 ")
print("==================================================")
display(summary_df)

# Sauvegarde de l'artefact pour le module analytics global du groupe
summary_df.to_csv("backend/artifacts/msd_vs_jaccard_comparison.csv", index=False)

   COMPARAISON GLOBALE : MSD VS JACCARD CUSTOM    

👉 Calcul de toutes les métriques pour : MSD...
--- Métriques d'Erreur Numérique ---
RMSE : 0.8317
MAE  : 0.6359
--- Métriques de Classement (Top-10 à un seuil de 3.5) ---
Precision@10 : 84.43%
Recall@10    : 4.74%
--- Métriques de Qualité du Catalogue (Top-10) ---
Couverture du Catalogue : 8.94% (781 films sur 8737)
--- Métriques de Comportement (Novelty@10) ---
Nouveauté Moyenne (Novelty@10) : 1.8516 bits
✓ Calcul terminé pour MSD.

👉 Calcul de toutes les métriques pour : JACARD...
--- Métriques d'Erreur Numérique ---
RMSE : 0.8328
MAE  : 0.6345
--- Métriques de Classement (Top-10 à un seuil de 3.5) ---
Precision@10 : 85.46%
Recall@10    : 4.92%
--- Métriques de Qualité du Catalogue (Top-10) ---
Couverture du Catalogue : 7.36% (643 films sur 8737)
--- Métriques de Comportement (Novelty@10) ---
Nouveauté Moyenne (Novelty@10) : 1.4010 bits
✓ Calcul terminé pour JACARD.

👉 Calcul de toutes les métriques pour : COSINE_JACCARD...
--- Métr

,Métrique,Valeur (User-Based MSD),Valeur (User-Based Jaccard),Valeur (User-Based Cosine Jaccard)
0,RMSE,0.8317,0.8328,0.8308
1,MAE,0.6359,0.6345,0.6330
2,Precision@10,0.8443,0.8546,0.8567
3,Recall@10,0.0474,0.0492,0.0493
4,Catalog Coverage,0.0894,0.0736,0.0736
5,Novelty@10,1.8516,1.4010,1.4156


### Explanation of the different values of k

In [ ]:
# Exemple de boucle rapide à tester dans ton notebook :
for top_k in [10, 20, 50]:
    print(f"--- ÉVALUATION POUR TOP-{top_k} ---")
    precision_recall_at_k(all_predictions, k=top_k, threshold=3.5)

--- ÉVALUATION POUR TOP-10 ---
--- Métriques de Classement (Top-10 à un seuil de 3.5) ---
Precision@10 : 84.43%
Recall@10    : 4.74%
--- ÉVALUATION POUR TOP-20 ---
--- Métriques de Classement (Top-20 à un seuil de 3.5) ---
Precision@20 : 83.70%
Recall@20    : 9.15%
--- ÉVALUATION POUR TOP-50 ---
--- Métriques de Classement (Top-50 à un seuil de 3.5) ---
Precision@50 : 81.71%
Recall@50    : 20.91%
